<a href="https://colab.research.google.com/github/leadministrador/mineria-datos-ii-proyecto-integrador/blob/main/notebooks/01_exploracion_landing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pyspark

In [2]:
from google.colab import files
subido = files.upload()

Saving cloud_provider_challenge_dataset_v1.zip to cloud_provider_challenge_dataset_v1 (1).zip


In [3]:
import zipfile, os
nombre_zip = list(subido.keys())[0]
with zipfile.ZipFile(nombre_zip) as z:
    z.extractall("/content/proyecto")
LANDING = "/content/proyecto/datalake/landing"
print(sorted(os.listdir(LANDING)))
print("JSONL:", len(os.listdir(LANDING + "/usage_events_stream")))

['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'usage_events_stream', 'users.csv']
JSONL: 120


In [4]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("proyecto-integrador")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
print("Spark:", spark.version)

Spark: 4.0.4


In [5]:
from pyspark.sql import functions as F

fuentes = ["customers_orgs", "users", "resources", "support_tickets",
           "marketing_touches", "nps_surveys", "billing_monthly"]
dfs = {}
for f in fuentes:
    df = (spark.read.option("header", True)
          .option("inferSchema", True)
          .csv(f"{LANDING}/{f}.csv"))
    dfs[f] = df
    nulos = {c: df.filter(F.col(c).isNull()).count() for c in df.columns}
    nulos = {c: n for c, n in nulos.items() if n > 0}
    print(f"{f}: {df.count()} filas, {len(df.columns)} columnas | nulos: {nulos}")

customers_orgs: 80 filas, 11 columnas | nulos: {'nps_score': 11}
users: 800 filas, 7 columnas | nulos: {'last_login': 139}
resources: 400 filas, 7 columnas | nulos: {'tags_json': 83}
support_tickets: 1000 filas, 8 columnas | nulos: {'resolved_at': 240, 'csat': 254}
marketing_touches: 1500 filas, 7 columnas | nulos: {}
nps_surveys: 92 filas, 4 columnas | nulos: {'nps_score': 19, 'comment': 10}
billing_monthly: 240 filas, 8 columnas | nulos: {'credits': 137}


In [6]:
eventos = spark.read.json(LANDING + "/usage_events_stream")
eventos.printSchema()
print("Eventos:", eventos.count())

root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)

Eventos: 43200


In [9]:
def contar(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

eventos.select(
    F.count("*").alias("total"),
    F.countDistinct("event_id").alias("ids_unicos"),
    contar(F.col("value").isNull()).alias("value_nulo"),
    contar(F.col("unit").isNull()).alias("unit_nulo"),
    contar(F.col("cost_usd_increment") < 0).alias("costo_negativo"),
    F.max("cost_usd_increment").alias("costo_max"),
    contar(F.col("genai_tokens").isNotNull()).alias("con_genai"),
    contar(F.col("carbon_kg").isNotNull()).alias("con_carbon"),
).show()

+-----+----------+----------+---------+--------------+---------+---------+----------+
|total|ids_unicos|value_nulo|unit_nulo|costo_negativo|costo_max|con_genai|con_carbon|
+-----+----------+----------+---------+--------------+---------+---------+----------+
|43200|     43200|       877|     2075|           216| 317.4308|     3132|     32400|
+-----+----------+----------+---------+--------------+---------+---------+----------+



In [7]:
(eventos.groupBy("schema_version")
    .agg(F.count("*").alias("eventos"),
         F.min("timestamp").alias("desde"),
         F.max("timestamp").alias("hasta"))
    .orderBy("schema_version")
    .show(truncate=False))

eventos.groupBy("service").count().orderBy(F.desc("count")).show()

+--------------+-------+--------------------+--------------------+
|schema_version|eventos|desde               |hasta               |
+--------------+-------+--------------------+--------------------+
|1             |10800  |2025-07-03T00:02:00Z|2025-07-17T23:56:00Z|
|2             |32400  |2025-07-18T00:01:00Z|2025-08-31T23:58:00Z|
+--------------+-------+--------------------+--------------------+

+----------+-----+
|   service|count|
+----------+-----+
|   compute|12498|
|   storage| 7614|
|  database| 7419|
|networking| 6921|
| analytics| 4590|
|     genai| 4158|
+----------+-----+



In [8]:
b = dfs["billing_monthly"]
b.groupBy("currency").count().show()
b.groupBy("month").count().orderBy("month").show()
print("Subtotal negativo:", b.filter(F.col("subtotal") < 0).count())

+--------+-----+
|currency|count|
+--------+-----+
|     ARS|   51|
|     USD|  160|
|     EUR|   29|
+--------+-----+

+----------+-----+
|     month|count|
+----------+-----+
|2025-06-01|   80|
|2025-07-01|   80|
|2025-08-01|   80|
+----------+-----+

Subtotal negativo: 13
